# Meta sentiment: scoring the posts not yet scored

Scores the 270,675 Meta posts that are not in meta_post_sentiment, with the same model and settings as the existing scores (cell 5 checks this). X needs nothing: all 540,225 X posts are already scored. Use a GPU runtime. Documented in docs/Geolocation_Extraction.docx, section 5.

## Cell 1: Google Drive and GPU check (any GPU works; an A100 is fastest)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount("/content/drive")

## Cell 2: connect to the database (same as notebooks/colab_x_corpus.ipynb)

In [ ]:
!pip install -q "cloud-sql-python-connector[pg8000]" sqlalchemy pandas
from google.colab import auth
auth.authenticate_user()     # sign in with your @g.harvard.edu account

import getpass
import pandas as pd
import sqlalchemy
from google.cloud.sql.connector import Connector

INSTANCE = "harvard-capstone-499102:us-central1:h5n1-pg"
DB_PASSWORD = getpass.getpass("h5n1_app password: ")
connector = Connector()

def _getconn():
    return connector.connect(INSTANCE, "pg8000", user="h5n1_app",
                             password=DB_PASSWORD, db="h5n1")

engine = sqlalchemy.create_engine("postgresql+pg8000://", creator=_getconn,
                                  pool_pre_ping=True)

## Cell 3: which posts still need a score

In [ ]:
# meta_post_sentiment holds the scores made so far (the 30,369 Candidates). Every other
# post in meta_post_analysis -- US or not -- is scored here. X needs nothing: all
# 540,225 X posts are already in x_post_sentiment.
todo_ids = pd.read_sql("""
    SELECT m.post_id
    FROM meta_post_analysis m
    WHERE NOT EXISTS (SELECT 1 FROM meta_post_sentiment s WHERE s.post_id = m.post_id)
    ORDER BY m.post_id""", engine).post_id.astype(str).tolist()
print(f"{len(todo_ids):,} Meta posts to score")

## Cell 4: the model: the same one, with the same settings, as the existing scores

In [ ]:
# Waree's notebook: cardiffnlp/twitter-xlm-roberta-base-sentiment, line breaks replaced
# by spaces, the first 512 tokens of each post, all three scores kept.
import torch
from transformers import pipeline

clf = pipeline("sentiment-analysis", model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
               top_k=None, device=0 if torch.cuda.is_available() else -1)

def score(texts, batch_size=64):
    """Texts -> DataFrame of negative / neutral / positive, in the order given."""
    cleaned = [(t or "").replace("\n", " ") for t in texts]
    res = clf(cleaned, batch_size=batch_size, truncation=True, max_length=512)
    return pd.DataFrame([{d["label"]: d["score"] for d in r} for r in res])[
        ["negative", "neutral", "positive"]]

## Cell 5: check: re-score 40 posts that already have scores; they should match

In [ ]:
check = pd.read_sql("""
    SELECT m.post_id, m.post_text, s.negative, s.neutral, s.positive
    FROM meta_post_sentiment s JOIN meta_post_analysis m USING (post_id)
    ORDER BY md5(m.post_id || 'repro') LIMIT 40""", engine)
new = score(check.post_text.tolist())
diff = (new.values - check[["negative", "neutral", "positive"]].values)
print(f"largest difference: {abs(diff).max():.6f} | same top label: "
      f"{(new.values.argmax(1) == check[['negative', 'neutral', 'positive']].values.argmax(1)).mean():.0%}")
# Expect a largest difference well under 0.01 (on a GPU the last decimals can differ
# slightly from the CPU) and the same top label for every post.

## Cell 6: score the rest in chunks of 5,000, each saved to Drive; resumes after a drop

In [ ]:
import time
from pathlib import Path

PARTS = Path("/content/drive/MyDrive/h5n1_sentiment/meta_parts")
PARTS.mkdir(parents=True, exist_ok=True)
CHUNK = 5000
chunks = [todo_ids[i:i + CHUNK] for i in range(0, len(todo_ids), CHUNK)]
t0, n_done = time.time(), 0
for k, ids in enumerate(chunks):
    part = PARTS / f"part_{k:04d}.csv"
    if part.exists():                                 # done in an earlier session
        continue
    posts = pd.read_sql(sqlalchemy.text(
        "SELECT post_id, post_text FROM meta_post_analysis WHERE post_id = ANY(:ids)"),
        engine, params={"ids": ids})
    posts["post_id"] = posts.post_id.astype(str)
    posts = posts.assign(n=posts.post_text.fillna("").str.len()).sort_values("n")  # similar
    out = score(posts.post_text.tolist())                                         # lengths
    out.insert(0, "post_id", posts.post_id.values)                               # batch faster
    out.to_csv(part, index=False)
    n_done += len(out)
    rate = n_done / (time.time() - t0)
    print(f"  part {k + 1}/{len(chunks)}  {rate:.0f} posts/s  "
          f"~{max(0, len(todo_ids) - CHUNK * (k + 1)) / rate / 60:.0f} min left")
print("all parts written")

## Cell 7: combine the parts and check them; add them to meta_post_sentiment when ready

In [ ]:
done = pd.concat([pd.read_csv(p, dtype={"post_id": str}) for p in sorted(PARTS.glob("part_*.csv"))],
                 ignore_index=True)
print(f"{len(done):,} posts scored | expected {len(todo_ids):,}")
assert done.post_id.is_unique, "a post was scored twice"
assert set(done.post_id) == set(todo_ids), "some posts are missing: rerun cell 6"
assert done[["negative", "neutral", "positive"]].notna().all().all()
print("score sums are 1:", ((done[["negative", "neutral", "positive"]].sum(axis=1) - 1).abs() < 1e-4).all())
print(done[["negative", "neutral", "positive"]].idxmax(axis=1).value_counts(normalize=True).round(3))
done.to_csv(PARTS.parent / "meta_sentiment_remaining.csv", index=False)

# The columns match meta_post_sentiment exactly (post_id, negative, neutral, positive),
# so the new rows can be appended to it. Writing to the shared database is a separate,
# deliberate step. When ready:
#   done.to_sql("meta_post_sentiment", engine, if_exists="append", index=False,
#               method="multi", chunksize=1000)
#   pd.read_sql("SELECT count(*) FROM meta_post_sentiment", engine)   # 301,044 afterwards